# Interactive Wildfire Visualization with `ipyleaflet`
**Advanced GIS Techniques — Package Demo**

---

All rasters were pre-processed in **ArcGIS**. This notebook is purely about **visualization with ipyleaflet**.

| File | Description |
|------|-------------|
| `Pre_TrueColor_50km.tif` | Pre-fire true color RGB |
| `Post_TrueColor_50km.tif` | Post-fire true color RGB |
| `Pre_NBR_50km.tif` | Pre-fire NBR |
| `Post_NBR_50km.tif` | Post-fire NBR |
| `dNBR_50km.tif` | Continuous dNBR |
| `dNBR_Severity_Class_50km.tif` | Classified burn severity (5 classes) |

##  Step 0 — Install & Import

In [ ]:
!pip install -q ipyleaflet rasterio pillow ipywidgets
print('Done')

Done


In [ ]:
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds
from rasterio.crs import CRS
import io, base64
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display
from ipyleaflet import (
    Map, ImageOverlay, LayersControl,
    WidgetControl, ScaleControl, basemap_to_tiles, basemaps
)
import warnings
warnings.filterwarnings('ignore')
print('Imports ready')

Imports ready


##  Step 1 — Upload Files from Google Drive or Local

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path to files stored in Drive
DATA_DIR = '/content/drive/MyDrive/GEE_Fire_Exports'
print(f' Drive mounted — looking for files in: {DATA_DIR}')

Mounted at /content/drive
 Drive mounted — looking for files in: /content/drive/MyDrive/GEE_Fire_Exports


In [ ]:
import os

for f in os.listdir(DATA_DIR):
    print(f)

PRE_20240830_RGB.tif
Pre_TrueColor_50km.tif
Post_TrueColor_50km.tif
Pre_NBR_50km.tif
Post_NBR_50km.tif
dNBR_50km.tif
dNBR_Severity_Class_50km.tif


In [ ]:
import os

FILES = {
    'pre_rgb':    os.path.join(DATA_DIR, 'Pre_TrueColor_50km.tif'),
    'post_rgb':   os.path.join(DATA_DIR, 'Post_TrueColor_50km.tif'),
    'pre_nbr':    os.path.join(DATA_DIR, 'Pre_NBR_50km.tif'),
    'post_nbr':   os.path.join(DATA_DIR, 'Post_NBR_50km.tif'),
    'dnbr':       os.path.join(DATA_DIR, 'dNBR_50km.tif'),
    'severity':   os.path.join(DATA_DIR, 'dNBR_Severity_Class_50km.tif'),
}

print('Checking files...')

all_ok = True

for key, path in FILES.items():

    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        status = f'{size:.1f} MB'
    else:
        status = ' NOT FOUND'
        all_ok = False

    print(f'{key:12s} {status} → {os.path.basename(path)}')

print('\n Ready!' if all_ok else '\n Fix missing paths above and re-run')

Checking files...
pre_rgb      672.4 MB → Pre_TrueColor_50km.tif
post_rgb     662.7 MB → Post_TrueColor_50km.tif
pre_nbr      118.9 MB → Pre_NBR_50km.tif
post_nbr     119.2 MB → Post_NBR_50km.tif
dnbr         117.6 MB → dNBR_50km.tif
severity     3.1 MB → dNBR_Severity_Class_50km.tif

 Ready!


## 🔄 Step 2 — Convert Rasters to Web Images

`ipyleaflet`'s `ImageOverlay` needs a PNG image with lat/lon bounds in **WGS84**.
This cell reads each GeoTIFF, reprojects if needed, and encodes it as a base64 PNG.

In [ ]:
with rasterio.open(FILES['dnbr']) as src:
    print(src.crs)

EPSG:4326


In [ ]:
TARGET_CRS = CRS.from_epsg(4326)   # WGS84 — required by Leaflet
MAX_PX = 1024                       # max dimension for overlay (keeps file size manageable)

def get_wgs84_bounds(path):
    """Return [[south, west], [north, east]] bounds in WGS84."""
    with rasterio.open(path) as src:
        bounds = transform_bounds(src.crs, TARGET_CRS, *src.bounds) \
                 if src.crs != TARGET_CRS else src.bounds
    left, bottom, right, top = bounds
    return [[bottom, left], [top, right]]

def reproject_array(path, band=1):
    """Read a single band, reproject to WGS84, return (array, profile)."""
    with rasterio.open(path) as src:
        if src.crs == TARGET_CRS:
            return src.read(band).astype(np.float32), src.profile
        transform, w, h = calculate_default_transform(
            src.crs, TARGET_CRS, src.width, src.height, *src.bounds)
        data = np.zeros((h, w), dtype=np.float32)
        reproject(rasterio.band(src, band), data,
                  src_transform=src.transform, src_crs=src.crs,
                  dst_transform=transform, dst_crs=TARGET_CRS,
                  resampling=Resampling.bilinear)
        profile = src.profile.copy()
        profile.update(crs=TARGET_CRS, transform=transform, width=w, height=h)
        return data, profile

def reproject_rgb(path):
    """Read 3-band RGB, reproject to WGS84, return (H,W,3) uint8 array."""
    with rasterio.open(path) as src:
        if src.crs == TARGET_CRS:
            rgb = src.read([1,2,3]).astype(np.float32)
            profile = src.profile
        else:
            transform, w, h = calculate_default_transform(
                src.crs, TARGET_CRS, src.width, src.height, *src.bounds)
            rgb = np.zeros((3, h, w), dtype=np.float32)
            for i, b in enumerate([1,2,3]):
                reproject(rasterio.band(src, b), rgb[i],
                          src_transform=src.transform, src_crs=src.crs,
                          dst_transform=transform, dst_crs=TARGET_CRS,
                          resampling=Resampling.bilinear)
            profile = src.profile.copy()
            profile.update(crs=TARGET_CRS, transform=transform, width=w, height=h)
    # Normalize each band to 0-255
    out = np.zeros_like(rgb, dtype=np.uint8)
    for i in range(3):
        band = rgb[i]
        valid = band[band != 0]
        if len(valid) == 0: continue
        lo, hi = np.percentile(valid, 2), np.percentile(valid, 98)
        out[i] = np.clip((band - lo) / (hi - lo + 1e-8) * 255, 0, 255).astype(np.uint8)
    return np.moveaxis(out, 0, -1)   # (H,W,3)

def thumbnail(arr_hw3_or_hw, max_px=MAX_PX):
    """Resize to fit within max_px on the longest side."""
    h, w = arr_hw3_or_hw.shape[:2]
    scale = max_px / max(h, w)
    if scale < 1:
        new_h, new_w = int(h * scale), int(w * scale)
        img = Image.fromarray(arr_hw3_or_hw)
        img = img.resize((new_w, new_h), Image.LANCZOS)
        return np.array(img)
    return arr_hw3_or_hw

def to_base64_png(rgba_arr):
    """Encode RGBA uint8 array as base64 PNG data URL."""
    img = Image.fromarray(rgba_arr.astype(np.uint8), 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='PNG', optimize=True)
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

def rgb_to_url(path, alpha=230):
    """Convert RGB GeoTIFF to base64 PNG URL."""
    arr = reproject_rgb(path)           # (H,W,3) uint8
    arr = thumbnail(arr)
    rgba = np.dstack([arr, np.full(arr.shape[:2], alpha, np.uint8)])
    return to_base64_png(rgba)

def single_band_to_url(path, cmap='RdYlGn', vmin=None, vmax=None, alpha=210):
    """Convert single-band GeoTIFF to colorized base64 PNG URL."""
    data, _ = reproject_array(path)
    data = thumbnail(data)
    valid = data[np.isfinite(data) & (data != 0)]
    lo = vmin if vmin is not None else np.percentile(valid, 2)
    hi = vmax if vmax is not None else np.percentile(valid, 98)
    norm = mcolors.Normalize(vmin=lo, vmax=hi)
    colormap = plt.get_cmap(cmap)
    rgba = colormap(norm(np.nan_to_num(data)))
    rgba[..., 3] = np.where(np.isfinite(data) & (data != 0), alpha / 255, 0)
    return to_base64_png((rgba * 255).astype(np.uint8))

# Severity class colormap (matches USGS standard colors)
SEVERITY_COLORS = {
    1: (0,   100,  0,  210),   # Enhanced Regrowth — dark green
    2: (144, 238, 144, 210),   # Unburned          — light green
    3: (255, 215,  0,  210),   # Low Severity      — yellow
    4: (255, 140,  0,  210),   # Moderate Severity — orange
    5: (139,   0,  0,  210),   # High Severity     — dark red
}

def severity_to_url(path):
    """Convert classified severity GeoTIFF to RGBA PNG URL."""
    data, _ = reproject_array(path, band=1)
    data = thumbnail(data.astype(np.int16))
    h, w = data.shape
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    for cls_val, color in SEVERITY_COLORS.items():
        mask = (data == cls_val)
        rgba[mask] = color
    return to_base64_png(rgba)

print('Helper functions ready')

Helper functions ready


In [ ]:
# ── Convert all layers  ───────────────
print('Converting layers — please wait...')

print('  [1/6] Pre-fire true color...')
url_pre_rgb   = rgb_to_url(FILES['pre_rgb'])

print('  [2/6] Post-fire true color...')
url_post_rgb  = rgb_to_url(FILES['post_rgb'])

print('  [3/6] Pre-fire NBR...')
url_pre_nbr   = single_band_to_url(FILES['pre_nbr'],  cmap='RdYlGn', vmin=-1, vmax=1)

print('  [4/6] Post-fire NBR...')
url_post_nbr  = single_band_to_url(FILES['post_nbr'], cmap='RdYlGn', vmin=-1, vmax=1)

print('  [5/6] dNBR continuous...')
url_dnbr      = single_band_to_url(FILES['dnbr'],     cmap='RdYlGn_r', vmin=-0.5, vmax=0.8)

print('  [6/6] Severity classification...')
url_severity  = severity_to_url(FILES['severity'])

# Bounds — use severity file (smallest, fastest to read)
BOUNDS = get_wgs84_bounds(FILES['severity'])
center_lat = (BOUNDS[0][0] + BOUNDS[1][0]) / 2
center_lon = (BOUNDS[0][1] + BOUNDS[1][1]) / 2

print(f'\n All layers ready!')
print(f'   Map center: ({center_lat:.4f}, {center_lon:.4f})')
print(f'   Bounds: {BOUNDS}')

Converting layers — please wait...
  [1/6] Pre-fire true color...
  [2/6] Post-fire true color...
  [3/6] Pre-fire NBR...
  [4/6] Post-fire NBR...
  [5/6] dNBR continuous...
  [6/6] Severity classification...

 All layers ready!
   Map center: (42.1359, -115.3419)
   Bounds: [[41.685961433452746, -115.94465540602252], [42.58589368508368, -114.73911629473412]]


---
#  DEMO — Building the Interactive Map

Each step adds one component. The map above updates live as you run each cell.

##  Step 1 — Create the Map Widget

In [ ]:
# Map() — the interactive Leaflet widget
# center = (lat, lon)  |  zoom = initial zoom level

m = Map(
    center=(center_lat, center_lon),
    zoom=9,
    scroll_wheel_zoom=True,
    layout=widgets.Layout(width='100%', height='520px')
)

display(m)
print(' Map widget created — interactive Leaflet map')

Map(center=[42.13592755926821, -115.34188585037832], controls=(ZoomControl(options=['position', 'zoom_in_text'…

 Map widget created — interactive Leaflet map


##  Demo Step 2 — Add Base Tile Layers

In [ ]:
# TileLayer — connects to a tile server for background imagery
# ipyleaflet has many built-in basemaps via basemap_to_tiles()

satellite = basemap_to_tiles(basemaps.Esri.WorldImagery)
satellite.name = ' Satellite'
satellite.base = True

topo = basemap_to_tiles(basemaps.OpenTopoMap)
topo.name = ' Terrain'
topo.base = True
topo.visible = False

m.add_layer(satellite)
m.add_layer(topo)

print('Satellite + terrain basemaps added')

Satellite + terrain basemaps added


###  Demo Step 3 — Add Raster Overlays

In [ ]:
# Define overlay layers
layers = [
    ('Pre-fire (True Color)', url_pre_rgb, True),
    ('Post-fire (True Color)', url_post_rgb, True),
    ('Pre-fire NBR', url_pre_nbr, True),
    ('Post-fire NBR', url_post_nbr, True),
    ('dNBR (continuous)', url_dnbr, True),
    ('Burn Severity (classified)', url_severity, True)
]

for name, url, visible in layers:
    layer = ImageOverlay(
        url=url,
        bounds=BOUNDS,
        name=name,
        opacity=0.9
    )

    layer.visible = visible   # set visibility explicitly
    m.add_layer(layer)

print('6 raster layers added')

6 raster layers added


## Demo Step 4 — Add Layer Control

In [ ]:
# LayersControl — toggle panel in the map corner
# Users can click to show/hide any layer

m.add_control(LayersControl(position='topright', collapsed=False))
m.add_control(ScaleControl(position='bottomleft'))

print(' LayersControl added — try toggling layers in the top-right panel')
print('Scale bar added — bottom-left')

 LayersControl added — try toggling layers in the top-right panel
Scale bar added — bottom-left


##  Demo Step 5 — Add Severity Legend

In [ ]:
# Simple legend for burn severity

legend_html = """
<div style="background:white;padding:8px;border-radius:5px;font-size:12px;">
<b>Burn Severity (dNBR)</b><br>
<span style="color:#006400">■</span> Enhanced Regrowth<br>
<span style="color:#90EE90">■</span> Unburned<br>
<span style="color:#FFD700">■</span> Low Severity<br>
<span style="color:#FF8C00">■</span> Moderate Severity<br>
<span style="color:#8B0000">■</span> High Severity
</div>
"""

m.add_control(WidgetControl(
    widget=widgets.HTML(value=legend_html),
    position='bottomright'
))

print('Legend added ')

Legend added 


##  Bonus — Opacity Slider

In [ ]:
# ipywidgets slider linked to the severity overlay opacity
# This shows how ipyleaflet integrates with the broader widget ecosystem

slider = widgets.FloatSlider(
    value=0.88, min=0.0, max=1.0, step=0.05,
    description='Severity opacity:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='380px')
)

def on_change(change):
    layer_severity.opacity = change['new']

slider.observe(on_change, names='value')

display(widgets.VBox([
    widgets.Label('  Drag to adjust severity layer opacity:'),
    slider
]))

print(' Slider linked to the severity layer, try it!')

 Slider linked to the severity layer, try it!


In [ ]:
display(m)

Map(bottom=97497.0, center=[42.07682027721137, -115.09552001953126], controls=(ZoomControl(options=['position'…

---
##  What We Built

| ipyleaflet class | What it does |
|------------------|--------------|
| `Map()` | Creates the interactive Leaflet canvas |
| `basemap_to_tiles()` | Adds satellite / terrain background |
| `ImageOverlay()` | Places georeferenced PNGs on the map |
| `LayersControl()` | Toggle panel to show/hide layers |
| `WidgetControl()` | Embeds HTML legend into the map |
| `ScaleControl()` | Adds a scale bar |
| `ipywidgets.FloatSlider` | Controls overlay opacity live |

**Docs:** https://ipyleaflet.readthedocs.io/en/latest/
**GitHub:** https://github.com/jupyter-widgets/ipyleaflet